# Dataset Load Tests
This notebook completely aggregates all dataset tests natively.

## Test: `test_iq_oth.py`
**Documentation**:
```text
Test program: IQ-OTH/NCCD Lung Cancer Dataset
Modality   : CT (Chest) — JPEG/PNG slices
Domain     : Radiology
Tools      : SimpleImageWrapper (TissueLab-SDK), PIL, numpy, matplotlib

Loads one sample from each of the three classes (Benign, Malignant, Normal),
extracts pixel-level statistics, renders a diagnostic visualization, and writes
a structured JSON report.
```

**Expected Input:** Sample image file loaded from the relative `week1/data` directory as resolved below.
**Expected Output:** Successful execution outputting pixel metrics and dumping JSON reports/matplotlib figures into `week1/output`.


In [ ]:
"""
Test program: IQ-OTH/NCCD Lung Cancer Dataset
Modality   : CT (Chest) — JPEG/PNG slices
Domain     : Radiology
Tools      : SimpleImageWrapper (TissueLab-SDK), PIL, numpy, matplotlib

Loads one sample from each of the three classes (Benign, Malignant, Normal),
extracts pixel-level statistics, renders a diagnostic visualization, and writes
a structured JSON report.
"""

import json
import sys
from pathlib import Path

import numpy as np
from PIL import Image

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("[WARN] matplotlib not installed — skipping visualizations")

try:
    from tissuelab_sdk.wrapper import SimpleImageWrapper
    HAS_SDK = True
except ImportError:
    HAS_SDK = False
    print("[WARN] TissueLab-SDK not installed — falling back to PIL only")

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
DATA_DIR = Path("week1/test_iq_oth.py").parent / "data" / "IQ-OTH_NCCD"
OUTPUT_DIR = Path("week1/test_iq_oth.py").parent / "output" / "IQ-OTH_NCCD"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLES: dict[str, Path] = {
    "Benign":    DATA_DIR / "Benign"    / "Benign_case_1.jpg",
    "Malignant": DATA_DIR / "Malignant" / "Malignant_case_1.jpg",
    "Normal":    DATA_DIR / "Normal"    / "Normal_case_1.jpg",
}

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def pixel_stats(arr: np.ndarray) -> dict:
    """Compute comprehensive pixel statistics from a numpy array."""
    flat = arr.flatten().astype(float)
    counts, edges = np.histogram(flat, bins=10)
    return {
        "shape":        list(arr.shape),
        "dtype":        str(arr.dtype),
        "min":          float(flat.min()),
        "max":          float(flat.max()),
        "mean":         round(float(flat.mean()), 4),
        "std":          round(float(flat.std()), 4),
        "median":       round(float(np.median(flat)), 4),
        "percentile_5": round(float(np.percentile(flat, 5)), 4),
        "percentile_95":round(float(np.percentile(flat, 95)), 4),
        "histogram_counts": counts.tolist(),
        "histogram_edges":  [round(e, 2) for e in edges.tolist()],
    }


def load_with_sdk(path: Path) -> dict:
    """Load image via SimpleImageWrapper and extract wrapper-level metadata."""
    wrapper = SimpleImageWrapper(str(path))
    info = {
        "wrapper_class":      type(wrapper).__name__,
        "dimensions_wh":      list(wrapper.dimensions),
        "level_count":        wrapper.level_count,
        "level_dimensions":   [list(d) for d in wrapper.level_dimensions],
        "properties":         dict(wrapper.properties),
    }
    # Read full region as array
    w, h = wrapper.dimensions
    region_arr = wrapper.read_region((0, 0), 0, (w, h), as_array=True)
    info["region_shape"] = list(region_arr.shape)
    wrapper.close()
    return info, region_arr


def load_with_pil(path: Path) -> tuple[dict, np.ndarray]:
    """Load image directly with PIL and extract format metadata."""
    img = Image.open(str(path))
    info = {
        "format":   img.format,
        "mode":     img.mode,
        "size_wh":  list(img.size),
        "info_tags": {k: str(v) for k, v in img.info.items()},
    }
    # Convert to RGB for consistent array shape
    rgb = img.convert("RGB")
    arr = np.array(rgb)
    img.close()
    return info, arr


# ---------------------------------------------------------------------------
# Per-sample analysis
# ---------------------------------------------------------------------------
def analyse_sample(label: str, path: Path) -> dict:
    print(f"\n{'='*60}")
    print(f"  CLASS: {label}")
    print(f"  FILE : {path.name}")
    print(f"{'='*60}")

    if not path.exists():
        print(f"  [ERROR] File not found: {path}")
        return {"label": label, "error": "file not found"}

    result: dict = {"label": label, "file": str(path)}

    # --- PIL load ---
    pil_info, arr = load_with_pil(path)
    result["pil"] = pil_info
    print(f"  PIL format  : {pil_info['format']}")
    print(f"  PIL mode    : {pil_info['mode']}")
    print(f"  Dimensions  : {pil_info['size_wh'][0]}W × {pil_info['size_wh'][1]}H")

    # --- SDK load ---
    if HAS_SDK:
        sdk_info, sdk_arr = load_with_sdk(path)
        result["sdk"] = sdk_info
        print(f"  SDK wrapper : {sdk_info['wrapper_class']}")
        print(f"  SDK dims    : {sdk_info['dimensions_wh']}")
        print(f"  Level count : {sdk_info['level_count']}")
        arr = sdk_arr  # prefer SDK-loaded array

    # --- Pixel statistics ---
    stats = pixel_stats(arr)
    result["pixel_stats"] = stats
    print(f"  Shape       : {stats['shape']}")
    print(f"  Dtype       : {stats['dtype']}")
    print(f"  Intensity   : min={stats['min']:.1f}  max={stats['max']:.1f}  "
          f"mean={stats['mean']:.1f}  std={stats['std']:.1f}")
    print(f"  Percentiles : p5={stats['percentile_5']}  p95={stats['percentile_95']}")

    # Per-channel stats for RGB
    if arr.ndim == 3 and arr.shape[2] == 3:
        channels = {}
        for i, ch in enumerate(["R", "G", "B"]):
            ch_arr = arr[:, :, i]
            channels[ch] = {
                "mean": round(float(ch_arr.mean()), 4),
                "std":  round(float(ch_arr.std()), 4),
            }
        result["channel_stats"] = channels
        print(f"  Channels    : R_mean={channels['R']['mean']}  "
              f"G_mean={channels['G']['mean']}  B_mean={channels['B']['mean']}")

    return result, arr


# ---------------------------------------------------------------------------
# Visualization
# ---------------------------------------------------------------------------
def make_visualization(samples: list[tuple[str, np.ndarray]]) -> None:
    if not HAS_MPL:
        return

    fig = plt.figure(figsize=(15, 10))
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

    for col, (label, arr) in enumerate(samples):
        # Image panel
        ax_img = fig.add_subplot(gs[0, col])
        ax_img.imshow(arr)
        ax_img.set_title(f"{label}", fontsize=13, fontweight="bold")
        ax_img.axis("off")

        # Histogram panel
        ax_hist = fig.add_subplot(gs[1, col])
        gray = np.mean(arr, axis=2) if arr.ndim == 3 else arr
        ax_hist.hist(gray.flatten(), bins=50, color="steelblue", edgecolor="none", alpha=0.8)
        ax_hist.set_xlabel("Pixel intensity", fontsize=9)
        ax_hist.set_ylabel("Count", fontsize=9)
        ax_hist.set_title(f"{label} — intensity histogram", fontsize=10)
        ax_hist.grid(True, alpha=0.3)

    fig.suptitle("IQ-OTH/NCCD Lung Cancer Dataset — Sample Analysis\n"
                 "CT chest slices: Benign · Malignant · Normal",
                 fontsize=14, fontweight="bold")

    out_path = OUTPUT_DIR / "IQ-OTH_NCCD_sample_analysis.png"
    fig.savefig(str(out_path), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"\n  [VIZ] Saved → {out_path}")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def main() -> None:
    print("\n" + "="*60)
    print("  IQ-OTH/NCCD Lung Cancer Dataset — Sample Probe")
    print("  Modality : Chest CT (JPEG slices)")
    print("  Domain   : Radiology")
    print("="*60)

    report: dict = {
        "dataset": "IQ-OTH/NCCD Lung Cancer Dataset",
        "modality": "CT (Chest)",
        "domain": "Radiology",
        "format": "JPEG",
        "classes": ["Benign", "Malignant", "Normal"],
        "clinical_context": (
            "2D CT chest slices labelled by Iraqi oncologists/radiologists. "
            "Benign=120 images/15 patients, Malignant=561/40, Normal=416/55. "
            "Original DICOM from SOMATOM Siemens; converted to JPEG at 1mm slice thickness."
        ),
        "samples": [],
    }

    arrays_for_viz: list[tuple[str, np.ndarray]] = []

    for label, path in SAMPLES.items():
        result, arr = analyse_sample(label, path)
        report["samples"].append(result)
        arrays_for_viz.append((label, arr))

    # Visualization
    make_visualization(arrays_for_viz)

    # Save JSON report
    report_path = OUTPUT_DIR / "IQ-OTH_NCCD_report.json"
    with open(report_path, "w") as f:
        json.dump(report, f, indent=2)

    print(f"\n[REPORT] Saved → {report_path}")
    print("\n[DONE]")


if __name__ == "__main__":
    main()


## Test: `test_oasis1.py`
**Documentation**:
```text
Test program: OASIS-1 Cross-Sectional Brain MRI Dataset
Modality   : MRI (T1-weighted MPRAGE, 3D structural brain)
Domain     : Radiology / Neuroscience
Format     : Analyze 7.5 (.hdr/.img pairs)
Tools      : nibabel (direct), NiftiImageWrapper (TissueLab-SDK), matplotlib, numpy

Subject    : OAS1_0001_MR1  (74-year-old female, CDR=0, MMSE=29 — cognitively normal)

What this program extracts:
  - Full subject metadata (age, sex, CDR, MMSE, brain volumes) from .txt file
  - Analyze 7.5 header info (voxel size, orientation, data type, field of view)
  - 3D volume statistics (intensity range, mean, std, tissue contrast)
  - Three-plane visualization (sagittal / coronal / axial) of the processed MRI
  - FSL tissue segmentation mask — per-label voxel counts (GM/WM/CSF)
  - NiftiImageWrapper region read (SDK interface demo)
  - JSON + PNG report output
```

**Expected Input:** Sample image file loaded from the relative `week1/data` directory as resolved below.
**Expected Output:** Successful execution outputting pixel metrics and dumping JSON reports/matplotlib figures into `week1/output`.


In [ ]:
"""
Test program: OASIS-1 Cross-Sectional Brain MRI Dataset
Modality   : MRI (T1-weighted MPRAGE, 3D structural brain)
Domain     : Radiology / Neuroscience
Format     : Analyze 7.5 (.hdr/.img pairs)
Tools      : nibabel (direct), NiftiImageWrapper (TissueLab-SDK), matplotlib, numpy

Subject    : OAS1_0001_MR1  (74-year-old female, CDR=0, MMSE=29 — cognitively normal)

What this program extracts:
  - Full subject metadata (age, sex, CDR, MMSE, brain volumes) from .txt file
  - Analyze 7.5 header info (voxel size, orientation, data type, field of view)
  - 3D volume statistics (intensity range, mean, std, tissue contrast)
  - Three-plane visualization (sagittal / coronal / axial) of the processed MRI
  - FSL tissue segmentation mask — per-label voxel counts (GM/WM/CSF)
  - NiftiImageWrapper region read (SDK interface demo)
  - JSON + PNG report output
"""

import json
import re
import sys
from pathlib import Path

import numpy as np

try:
    import nibabel as nib
    HAS_NIBABEL = True
except ImportError:
    print("[ERROR] nibabel is required: pip install nibabel")
    sys.exit(1)

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("[WARN] matplotlib not installed — skipping visualizations")

try:
    from tissuelab_sdk.wrapper import NiftiImageWrapper
    HAS_SDK = True
except ImportError:
    HAS_SDK = False
    print("[WARN] TissueLab-SDK not installed — skipping NiftiImageWrapper demo")

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
SUBJECT_ID  = "OAS1_0001_MR1"
DATA_DIR    = Path("week1/test_oasis1.py").parent / "data" / "Oasis1" / SUBJECT_ID
OUTPUT_DIR  = Path("week1/test_oasis1.py").parent / "output" / "Oasis1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_HDR         = DATA_DIR / "RAW" / f"{SUBJECT_ID}_mpr-1_anon.hdr"
PROCESSED_HDR   = DATA_DIR / "PROCESSED" / "MPRAGE" / "T88_111" / \
                  f"{SUBJECT_ID}_mpr_n4_anon_111_t88_masked_gfc.hdr"
FSEG_HDR        = DATA_DIR / "FSL_SEG" / \
                  f"{SUBJECT_ID}_mpr_n4_anon_111_t88_masked_gfc_fseg.hdr"
FSEG_TXT        = DATA_DIR / "FSL_SEG" / \
                  f"{SUBJECT_ID}_mpr_n4_anon_111_t88_masked_gfc_fseg.txt"
METADATA_TXT    = DATA_DIR / f"{SUBJECT_ID}.txt"

# FSL tissue labels (standard FAST 3-class segmentation)
FSEG_LABELS = {0: "Background", 1: "CSF", 2: "Gray Matter", 3: "White Matter"}

# ---------------------------------------------------------------------------
# Metadata parsing
# ---------------------------------------------------------------------------
def parse_metadata_txt(path: Path) -> dict:
    """Parse OASIS per-subject metadata text file into a dict."""
    if not path.exists():
        return {"error": f"metadata file not found: {path}"}

    meta: dict = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("SCAN") or line.startswith("TYPE") or \
               line.startswith("Vox") or line.startswith("Rect") or \
               line.startswith("Orient") or line.startswith("TR") or \
               line.startswith("TE") or line.startswith("TI") or \
               line.startswith("Flip") or line.startswith("mpr-"):
                continue
            if ":" in line:
                key, _, val = line.partition(":")
                meta[key.strip()] = val.strip()
    return meta


def parse_fseg_txt(path: Path) -> dict:
    """Parse FSL segmentation stats text file (tissue volumes)."""
    if not path.exists():
        return {}
    stats: dict = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) >= 2:
                stats[parts[0]] = parts[1]
    return stats

# ---------------------------------------------------------------------------
# Volume statistics
# ---------------------------------------------------------------------------
def volume_stats(data: np.ndarray, label: str) -> dict:
    """Compute statistics on a 3D volume."""
    flat = data.flatten()
    nonzero = flat[flat > 0]
    print(f"\n  [{label}]")
    print(f"    Shape        : {data.shape}")
    print(f"    Voxel count  : {flat.size:,}")
    print(f"    Non-zero     : {len(nonzero):,}  ({100*len(nonzero)/flat.size:.1f}%)")
    print(f"    Intensity    : min={flat.min():.1f}  max={flat.max():.1f}  "
          f"mean={flat.mean():.2f}  std={flat.std():.2f}")
    if len(nonzero):
        print(f"    Brain only   : mean={nonzero.mean():.2f}  std={nonzero.std():.2f}")

    return {
        "shape":           list(data.shape),
        "voxel_count":     int(flat.size),
        "nonzero_voxels":  int(len(nonzero)),
        "min":             float(flat.min()),
        "max":             float(flat.max()),
        "mean":            round(float(flat.mean()), 4),
        "std":             round(float(flat.std()), 4),
        "brain_mean":      round(float(nonzero.mean()), 4) if len(nonzero) else None,
        "brain_std":       round(float(nonzero.std()), 4) if len(nonzero) else None,
    }


def analyze_header(img: nib.analyze.AnalyzeImage) -> dict:
    """Extract key header fields from an Analyze 7.5 / NIfTI header."""
    hdr = img.header
    zooms = hdr.get_zooms()
    return {
        "format":           type(img).__name__,
        "data_shape":       list(hdr.get_data_shape()),
        "voxel_size_mm":    [round(float(z), 4) for z in zooms],
        "data_dtype":       str(hdr.get_data_dtype()),
        "slope_intercept":  [float(hdr.get_slope_inter()[0] or 1.0),
                             float(hdr.get_slope_inter()[1] or 0.0)],
        "affine":           img.affine.tolist(),
        "header_str":       str(hdr),
    }


# ---------------------------------------------------------------------------
# Visualization
# ---------------------------------------------------------------------------
def make_visualization(processed_data: np.ndarray, fseg_data: np.ndarray | None) -> None:
    if not HAS_MPL:
        return

    # Pick middle slices in each orientation
    processed_data = np.squeeze(processed_data)
    sx, sy, sz = processed_data.shape
    sag_slice = processed_data[sx // 2, :, :]
    cor_slice = processed_data[:, sy // 2, :]
    axi_slice = processed_data[:, :, sz // 2]

    rows = 2 if fseg_data is not None else 1
    fig, axes = plt.subplots(rows, 3, figsize=(15, 5 * rows))
    if rows == 1:
        axes = axes[np.newaxis, :]

    for ax, sl, title in zip(axes[0],
                              [sag_slice, cor_slice, axi_slice],
                              ["Sagittal (mid)", "Coronal (mid)", "Axial (mid)"]):
        ax.imshow(np.rot90(sl), cmap="gray", interpolation="lanczos")
        ax.set_title(f"T88 Processed MRI — {title}", fontsize=11)
        ax.axis("off")

    if fseg_data is not None:
        fseg_data = np.squeeze(fseg_data)
        fsx, fsy, fsz = fseg_data.shape
        f_sag = fseg_data[fsx // 2, :, :]
        f_cor = fseg_data[:, fsy // 2, :]
        f_axi = fseg_data[:, :, fsz // 2]
        cmap = plt.cm.get_cmap("Set1", 4)
        for ax, sl, title in zip(axes[1],
                                  [f_sag, f_cor, f_axi],
                                  ["Sagittal", "Coronal", "Axial"]):
            ax.imshow(np.rot90(sl), cmap=cmap, vmin=0, vmax=3, interpolation="nearest")
            ax.set_title(f"FSL Segmentation — {title}", fontsize=11)
            ax.axis("off")

        # Add colorbar legend
        import matplotlib.patches as mpatches
        colors = [cmap(i) for i in range(4)]
        patches = [mpatches.Patch(color=colors[i], label=FSEG_LABELS[i]) for i in range(4)]
        fig.legend(handles=patches, loc="lower center", ncol=4, fontsize=10,
                   bbox_to_anchor=(0.5, -0.02))

    fig.suptitle(f"OASIS-1 Subject: {SUBJECT_ID}\n"
                 "T1-weighted MPRAGE brain MRI (T88 atlas space)",
                 fontsize=13, fontweight="bold")

    out_path = OUTPUT_DIR / f"{SUBJECT_ID}_visualization.png"
    fig.savefig(str(out_path), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"\n  [VIZ] Saved → {out_path}")


# ---------------------------------------------------------------------------
# SDK demo
# ---------------------------------------------------------------------------
def sdk_region_demo(hdr_path: Path) -> dict | None:
    if not HAS_SDK:
        return None
    print(f"\n  [SDK] Loading via NiftiImageWrapper: {hdr_path.name}")
    try:
        wrapper = NiftiImageWrapper(str(hdr_path))
        w, h = wrapper.dimensions
        region = wrapper.read_region((0, 0), 0, (w, h), as_array=True)
        info = {
            "wrapper_class":   type(wrapper).__name__,
            "dimensions_wh":   list(wrapper.dimensions),
            "level_count":     wrapper.level_count,
            "level_dimensions": [list(d) for d in wrapper.level_dimensions],
            "region_shape":    list(region.shape),
            "properties":      dict(wrapper.properties),
        }
        print(f"    Dimensions : {info['dimensions_wh']}")
        print(f"    Levels     : {info['level_count']}")
        print(f"    Region arr : {info['region_shape']}")
        wrapper.close()
        return info
    except Exception as e:
        print(f"    [WARN] NiftiImageWrapper failed: {e}")
        return {"error": str(e)}


# ---------------------------------------------------------------------------
# FSL segmentation analysis
# ---------------------------------------------------------------------------
def analyse_segmentation(fseg_path: Path) -> dict:
    """Load FSL tissue segmentation and compute per-label volume statistics."""
    if not fseg_path.exists():
        return {"error": f"FSL segmentation not found: {fseg_path}"}

    img = nib.load(str(fseg_path))
    data = np.asarray(img.get_fdata(), dtype=int)
    zooms = img.header.get_zooms()
    voxel_vol_mm3 = float(np.prod(zooms[:3]))

    label_stats: dict = {}
    for label_id, label_name in FSEG_LABELS.items():
        mask = data == label_id
        voxels = int(mask.sum())
        vol_cm3 = round(voxels * voxel_vol_mm3 / 1000.0, 4)
        label_stats[label_name] = {"voxels": voxels, "volume_cm3": vol_cm3}
        print(f"    {label_name:<15}: {voxels:>8,} voxels  ({vol_cm3:.2f} cm³)")

    return {
        "voxel_size_mm": [round(float(z), 4) for z in zooms[:3]],
        "voxel_vol_mm3": round(voxel_vol_mm3, 4),
        "tissue_volumes": label_stats,
    }


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def main() -> None:
    print("\n" + "="*60)
    print("  OASIS-1 Brain MRI Dataset — Sample Probe")
    print(f"  Subject  : {SUBJECT_ID}")
    print("  Modality : T1-weighted MPRAGE (3D structural brain MRI)")
    print("  Format   : Analyze 7.5 (.hdr/.img)")
    print("  Domain   : Radiology / Neuroscience")
    print("="*60)

    report: dict = {
        "dataset":   "OASIS-1 Cross-Sectional MRI",
        "subject":   SUBJECT_ID,
        "modality":  "T1-weighted MPRAGE (3D structural brain MRI)",
        "format":    "Analyze 7.5 (.hdr/.img)",
        "domain":    "Radiology / Neuroscience",
        "clinical_context": (
            "436-subject cross-sectional brain MRI study (Washington University). "
            "CDR 0=normal, 0.5=very mild dementia, 1=mild, 2=moderate. "
            "OAS1_0001_MR1: Female, 74y, CDR=0, MMSE=29 — cognitively normal."
        ),
    }

    # -- Metadata -----------------------------------------------------------
    print("\n--- Subject Metadata ---")
    meta = parse_metadata_txt(METADATA_TXT)
    report["subject_metadata"] = meta
    for k, v in meta.items():
        print(f"  {k:<12}: {v}")

    # -- RAW volume ---------------------------------------------------------
    print("\n--- RAW Acquire Volume (mpr-1) ---")
    if RAW_HDR.exists():
        raw_img = nib.load(str(RAW_HDR))
        raw_data = raw_img.get_fdata()
        raw_hdr_info = analyze_header(raw_img)
        raw_stats = volume_stats(raw_data, "RAW mpr-1")
        report["raw_volume"] = {"header": raw_hdr_info, "stats": raw_stats}
    else:
        print(f"  [WARN] RAW HDR not found: {RAW_HDR}")

    # -- Processed T88 volume -----------------------------------------------
    print("\n--- Processed T88 Atlas Volume ---")
    proc_data = None
    if PROCESSED_HDR.exists():
        proc_img = nib.load(str(PROCESSED_HDR))
        proc_data = proc_img.get_fdata()
        proc_hdr_info = analyze_header(proc_img)
        proc_stats = volume_stats(proc_data, "T88 Masked GFC")
        report["processed_volume"] = {"header": proc_hdr_info, "stats": proc_stats}

        # SDK demo on processed volume
        sdk_info = sdk_region_demo(PROCESSED_HDR)
        if sdk_info:
            report["sdk_nifti_wrapper"] = sdk_info
    else:
        print(f"  [WARN] Processed HDR not found: {PROCESSED_HDR}")

    # -- FSL Segmentation ---------------------------------------------------
    print("\n--- FSL Tissue Segmentation (GM / WM / CSF) ---")
    fseg_data = None
    if FSEG_HDR.exists():
        fseg_img = nib.load(str(FSEG_HDR))
        fseg_data = np.asarray(fseg_img.get_fdata(), dtype=int)
        seg_stats = analyse_segmentation(FSEG_HDR)
        report["fsl_segmentation"] = seg_stats

        # Also parse the FSL stats text if present
        fseg_txt_stats = parse_fseg_txt(FSEG_TXT)
        if fseg_txt_stats:
            report["fsl_segmentation"]["fseg_txt"] = fseg_txt_stats
    else:
        print(f"  [WARN] FSL segmentation not found: {FSEG_HDR}")

    # -- Visualization ------------------------------------------------------
    if proc_data is not None:
        make_visualization(proc_data, fseg_data)
    elif RAW_HDR.exists():
        make_visualization(raw_data, fseg_data)

    # -- Save report --------------------------------------------------------
    report_path = OUTPUT_DIR / f"{SUBJECT_ID}_report.json"
    with open(report_path, "w") as f:
        json.dump(report, f, indent=2)

    print(f"\n[REPORT] Saved → {report_path}")
    print("\n[DONE]")


if __name__ == "__main__":
    main()


## Test: `test_pkg_hsi.py`
**Documentation**:
```text
Test program: PKG – HistologyHSI-GB (Hyperspectral Histology of Glioblastoma)
Modality   : Hyperspectral Imaging (HSI) — H&E-stained histology slides
Domain     : Pathology (Brain tumor — GBM)
Format     : ENVI BIL (headerless binary + .hdr) + PNG preview
Tools      : spectral (Python Spectral Imagery), SimpleImageWrapper (TissueLab-SDK),
             numpy, matplotlib

Sample     : P1 / ROI_01_C01_T  (Patient 1, ROI 1, Tumor tissue)

What this program extracts:
  - Full ENVI header metadata (bands, wavelengths, sensor, interleave)
  - Raw hyperspectral cube shape: 800 × 1004 × 826 bands (400–1000 nm)
  - White/dark calibration references (reflectance normalisation)
  - Calibrated reflectance cube (pixel-level normalisation)
  - Spectral signature at multiple representative pixels
  - Per-band statistics (sample 20 bands across visible + NIR)
  - RGB preview via SimpleImageWrapper
  - False-colour band image (single band as grayscale)
  - Spectral mean plot
  - JSON + PNG report output
```

**Expected Input:** Sample image file loaded from the relative `week1/data` directory as resolved below.
**Expected Output:** Successful execution outputting pixel metrics and dumping JSON reports/matplotlib figures into `week1/output`.


In [ ]:
"""
Test program: PKG – HistologyHSI-GB (Hyperspectral Histology of Glioblastoma)
Modality   : Hyperspectral Imaging (HSI) — H&E-stained histology slides
Domain     : Pathology (Brain tumor — GBM)
Format     : ENVI BIL (headerless binary + .hdr) + PNG preview
Tools      : spectral (Python Spectral Imagery), SimpleImageWrapper (TissueLab-SDK),
             numpy, matplotlib

Sample     : P1 / ROI_01_C01_T  (Patient 1, ROI 1, Tumor tissue)

What this program extracts:
  - Full ENVI header metadata (bands, wavelengths, sensor, interleave)
  - Raw hyperspectral cube shape: 800 × 1004 × 826 bands (400–1000 nm)
  - White/dark calibration references (reflectance normalisation)
  - Calibrated reflectance cube (pixel-level normalisation)
  - Spectral signature at multiple representative pixels
  - Per-band statistics (sample 20 bands across visible + NIR)
  - RGB preview via SimpleImageWrapper
  - False-colour band image (single band as grayscale)
  - Spectral mean plot
  - JSON + PNG report output
"""

import json
import sys
from pathlib import Path

import numpy as np

try:
    import spectral
    import spectral.io.envi as envi
    HAS_SPECTRAL = True
except ImportError:
    print("[ERROR] spectral library is required: pip install spectral")
    sys.exit(1)

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("[WARN] matplotlib not installed — skipping visualizations")

try:
    from tissuelab_sdk.wrapper import SimpleImageWrapper
    HAS_SDK = True
except ImportError:
    HAS_SDK = False
    print("[WARN] TissueLab-SDK not available — skipping SDK demo")

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
PATIENT_ID = "P1"
ROI_ID     = "ROI_01_C01_T"   # Tumor ROI
LABEL      = "Tumor"

DATA_DIR   = Path("week1/test_pkg_hsi.py").parent / "data" / "PKG_HistologyHSI_GB" / PATIENT_ID / ROI_ID
OUTPUT_DIR = Path("week1/test_pkg_hsi.py").parent / "output" / "PKG_HistologyHSI_GB"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_HDR         = DATA_DIR / "raw.hdr"
RAW_BIN         = DATA_DIR / "raw"
WHITE_HDR       = DATA_DIR / "whiteReference.hdr"
WHITE_BIN       = DATA_DIR / "whiteReference"
DARK_HDR        = DATA_DIR / "darkReference.hdr"
DARK_BIN        = DATA_DIR / "darkReference"
RGB_PREVIEW     = DATA_DIR / "rgb.png"
PATIENT_PREVIEW = DATA_DIR.parent / f"{PATIENT_ID}.png"

# Wavelength range label
WL_RANGE = "400–1000 nm (826 bands)"

# ---------------------------------------------------------------------------
# Header parsing helper
# ---------------------------------------------------------------------------
def parse_envi_header(hdr_path: Path) -> dict:
    """Load ENVI header and return a serialisable metadata dict."""
    img = envi.open(str(hdr_path))
    md  = img.metadata
    # Extract wavelengths as floats
    wavelengths = None
    if "wavelength" in md:
        try:
            wavelengths = [float(w) for w in md["wavelength"]]
        except (ValueError, TypeError):
            wavelengths = md["wavelength"]

    result = {
        "lines":           int(md.get("lines", 0)),
        "samples":         int(md.get("samples", 0)),
        "bands":           int(md.get("bands", 0)),
        "interleave":      md.get("interleave", ""),
        "data_type":       md.get("data_type", ""),
        "byte_order":      md.get("byte_order", ""),
        "header_offset":   md.get("header offset", ""),
        "sensor_type":     md.get("sensor type", ""),
        "wavelength_units":md.get("wavelength units", ""),
        "wavelength_min":  round(wavelengths[0], 2) if wavelengths else None,
        "wavelength_max":  round(wavelengths[-1], 2) if wavelengths else None,
        "num_wavelengths": len(wavelengths) if wavelengths else None,
        "wavelength_sample_10": (
            [round(w, 2) for w in wavelengths[::len(wavelengths)//10]]
            if wavelengths and len(wavelengths) >= 10 else wavelengths
        ),
        "full_metadata": {k: str(v)[:200] for k, v in md.items()},
    }
    return result, img, wavelengths


# ---------------------------------------------------------------------------
# Calibration (reflectance = (raw - dark) / (white - dark))
# ---------------------------------------------------------------------------
def load_cube(hdr_path: Path) -> np.ndarray:
    """Load an ENVI file as a float32 numpy array (lines × samples × bands)."""
    img = envi.open(str(hdr_path))
    return img.load().astype(np.float32)


def calibrate(raw: np.ndarray, white: np.ndarray, dark: np.ndarray) -> np.ndarray:
    """
    Convert raw DN values to relative reflectance.
    white and dark are (1 × samples × bands) reference frames.
    """
    denom = white - dark
    # Avoid division by zero
    denom = np.where(denom == 0, 1e-6, denom)
    reflectance = (raw - dark) / denom
    return np.clip(reflectance, 0.0, 1.0)


# ---------------------------------------------------------------------------
# Spectral signature extraction
# ---------------------------------------------------------------------------
def spectral_signature(cube: np.ndarray, row: int, col: int, wavelengths: list) -> dict:
    """Extract and describe the spectral signature at a pixel."""
    sig = cube[row, col, :].tolist()
    wls = wavelengths or list(range(len(sig)))
    peak_idx = int(np.argmax(sig))
    return {
        "pixel_rc":   [row, col],
        "signature":  [round(float(v), 6) for v in sig],
        "peak_wl_nm": round(wls[peak_idx], 2),
        "peak_value": round(float(sig[peak_idx]), 6),
        "mean":       round(float(np.mean(sig)), 6),
        "std":        round(float(np.std(sig)), 6),
    }


def band_statistics(cube: np.ndarray, wavelengths: list, n_sample: int = 20) -> list:
    """Sample n bands evenly across the spectral dimension and compute stats."""
    n_bands = cube.shape[2]
    indices = np.linspace(0, n_bands - 1, n_sample, dtype=int)
    stats = []
    for idx in indices:
        band = cube[:, :, idx].flatten().astype(float)
        wl   = round(float(wavelengths[idx]), 2) if wavelengths else int(idx)
        stats.append({
            "band_index":    int(idx),
            "wavelength_nm": wl,
            "min":           round(float(band.min()), 6),
            "max":           round(float(band.max()), 6),
            "mean":          round(float(band.mean()), 6),
            "std":           round(float(band.std()), 6),
        })
    return stats


# ---------------------------------------------------------------------------
# Visualization
# ---------------------------------------------------------------------------
def make_visualization(
    raw_cube: np.ndarray,
    cal_cube: np.ndarray,
    wavelengths: list,
    rgb_path: Path,
    sdk_info: dict | None,
) -> None:
    if not HAS_MPL:
        return

    n_bands = raw_cube.shape[2]
    # Pick representative bands: ~450nm (blue), ~550nm (green), ~700nm (red-edge), ~800nm (NIR)
    if wavelengths:
        wl_arr = np.array(wavelengths)
        band_blue  = int(np.argmin(np.abs(wl_arr - 450)))
        band_green = int(np.argmin(np.abs(wl_arr - 550)))
        band_red   = int(np.argmin(np.abs(wl_arr - 650)))
        band_nir   = int(np.argmin(np.abs(wl_arr - 800)))
    else:
        band_blue, band_green, band_red, band_nir = 0, n_bands//4, n_bands//2, 3*n_bands//4

    def norm(arr2d: np.ndarray) -> np.ndarray:
        lo, hi = arr2d.min(), arr2d.max()
        return ((arr2d - lo) / (hi - lo + 1e-9) * 255).astype(np.uint8)

    # Build pseudo-RGB from calibrated cube
    r_ch = norm(cal_cube[:, :, band_red])
    g_ch = norm(cal_cube[:, :, band_green])
    b_ch = norm(cal_cube[:, :, band_blue])
    pseudo_rgb = np.stack([r_ch, g_ch, b_ch], axis=2)

    fig, axes = plt.subplots(2, 3, figsize=(16, 10))

    # 1. SDK / PIL RGB preview
    ax = axes[0, 0]
    if rgb_path.exists():
        from PIL import Image as PILImage
        preview = np.array(PILImage.open(str(rgb_path)).convert("RGB"))
        ax.imshow(preview)
        ax.set_title(f"RGB Preview (rgb.png)\n{PATIENT_ID}/{ROI_ID}", fontsize=10)
    else:
        ax.text(0.5, 0.5, "RGB preview\nnot available", ha="center", va="center",
                transform=ax.transAxes)
    ax.axis("off")

    # 2. Pseudo-RGB from calibrated reflectance
    axes[0, 1].imshow(pseudo_rgb)
    axes[0, 1].set_title(
        f"Pseudo-RGB (calibrated reflectance)\n"
        f"B={wavelengths[band_blue]:.0f}nm  G={wavelengths[band_green]:.0f}nm  "
        f"R={wavelengths[band_red]:.0f}nm",
        fontsize=10
    )
    axes[0, 1].axis("off")

    # 3. NIR band (single band grayscale)
    axes[0, 2].imshow(cal_cube[:, :, band_nir], cmap="viridis")
    axes[0, 2].set_title(
        f"NIR band — {wavelengths[band_nir]:.0f} nm\n(calibrated reflectance)", fontsize=10)
    axes[0, 2].axis("off")

    # 4. Spectral mean ± std across whole image
    ax4 = axes[1, 0]
    spatial_mean = cal_cube.reshape(-1, n_bands).mean(axis=0)
    spatial_std  = cal_cube.reshape(-1, n_bands).std(axis=0)
    wl_x = wavelengths if wavelengths else list(range(n_bands))
    ax4.fill_between(wl_x,
                     spatial_mean - spatial_std,
                     spatial_mean + spatial_std,
                     alpha=0.3, color="steelblue", label="±1 std")
    ax4.plot(wl_x, spatial_mean, color="steelblue", linewidth=1.5, label="mean")
    ax4.set_xlabel("Wavelength (nm)", fontsize=9)
    ax4.set_ylabel("Reflectance", fontsize=9)
    ax4.set_title("Mean spectral signature — full image", fontsize=10)
    ax4.legend(fontsize=8)
    ax4.grid(True, alpha=0.3)

    # 5. Three pixel signatures: center, corner, edge
    ax5 = axes[1, 1]
    h, w = cal_cube.shape[:2]
    pixels = [
        ("Center",     h // 2, w // 2, "tomato"),
        ("Top-left",   h // 8, w // 8, "seagreen"),
        ("Mid-right",  h // 2, 7*w//8, "mediumpurple"),
    ]
    for name, r, c, color in pixels:
        sig = cal_cube[r, c, :]
        ax5.plot(wl_x, sig, label=f"{name} ({r},{c})", color=color, linewidth=1.2)
    ax5.set_xlabel("Wavelength (nm)", fontsize=9)
    ax5.set_ylabel("Reflectance", fontsize=9)
    ax5.set_title("Pixel spectral signatures", fontsize=10)
    ax5.legend(fontsize=8)
    ax5.grid(True, alpha=0.3)

    # 6. Band variance map (which bands show the most spatial variance)
    ax6 = axes[1, 2]
    band_variance = cal_cube.reshape(-1, n_bands).var(axis=0)
    ax6.plot(wl_x, band_variance, color="darkorange", linewidth=1.2)
    ax6.set_xlabel("Wavelength (nm)", fontsize=9)
    ax6.set_ylabel("Spatial variance", fontsize=9)
    ax6.set_title("Per-band spatial variance\n(discriminative spectral regions)", fontsize=10)
    ax6.grid(True, alpha=0.3)

    fig.suptitle(
        f"PKG HistologyHSI-GB — {PATIENT_ID}/{ROI_ID} [{LABEL}]\n"
        "Hyperspectral Histology of Glioblastoma (H&E, 20×, 826 bands 400–1000 nm)",
        fontsize=13, fontweight="bold"
    )

    out_path = OUTPUT_DIR / f"{PATIENT_ID}_{ROI_ID}_analysis.png"
    fig.savefig(str(out_path), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"\n  [VIZ] Saved → {out_path}")


# ---------------------------------------------------------------------------
# SDK demo
# ---------------------------------------------------------------------------
def sdk_rgb_demo(png_path: Path) -> dict | None:
    if not HAS_SDK or not png_path.exists():
        return None
    print(f"\n  [SDK] Loading RGB preview via SimpleImageWrapper")
    wrapper = SimpleImageWrapper(str(png_path))
    w, h = wrapper.dimensions
    region = wrapper.read_region((0, 0), 0, (w, h), as_array=True)
    info = {
        "wrapper_class":  type(wrapper).__name__,
        "dimensions_wh":  list(wrapper.dimensions),
        "properties":     dict(wrapper.properties),
        "region_shape":   list(region.shape),
    }
    print(f"    Dimensions : {info['dimensions_wh']}")
    print(f"    Mode       : {wrapper.properties.get('mode', '?')}")
    wrapper.close()
    return info


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def main() -> None:
    print("\n" + "="*60)
    print("  PKG HistologyHSI-GB — Hyperspectral Histology Probe")
    print(f"  Sample   : {PATIENT_ID}/{ROI_ID}  [{LABEL}]")
    print("  Modality : Hyperspectral Imaging (HSI)")
    print("  Format   : ENVI BIL  (raw + raw.hdr)")
    print("  Domain   : Pathology (Glioblastoma)")
    print("="*60)

    report: dict = {
        "dataset":   "PKG HistologyHSI-GB",
        "patient":   PATIENT_ID,
        "roi":       ROI_ID,
        "label":     LABEL,
        "modality":  "Hyperspectral Imaging (HSI)",
        "format":    "ENVI BIL",
        "domain":    "Pathology (Brain — Glioblastoma)",
        "clinical_context": (
            "13-patient hyperspectral histology dataset of GBM tissue. "
            f"ROI {ROI_ID} is a Tumor (_T) region from Patient 1. "
            "826 spectral bands (400–1000 nm), spatial resolution 800×1004, "
            "captured at 20× magnification with HEADWALL Hyperspec III sensor."
        ),
    }

    # -- ENVI Header --------------------------------------------------------
    print("\n--- ENVI Header (raw cube) ---")
    if not RAW_HDR.exists():
        print(f"  [ERROR] raw.hdr not found: {RAW_HDR}")
        sys.exit(1)

    hdr_info, raw_img, wavelengths = parse_envi_header(RAW_HDR)
    report["envi_header"] = hdr_info
    print(f"  Lines × Samples × Bands : {hdr_info['lines']} × {hdr_info['samples']} × {hdr_info['bands']}")
    print(f"  Interleave              : {hdr_info['interleave']}")
    print(f"  Wavelength range        : {hdr_info['wavelength_min']} – {hdr_info['wavelength_max']} nm")
    print(f"  Sensor                  : {hdr_info.get('sensor_type', 'N/A')}")

    # -- White reference header
    if WHITE_HDR.exists():
        white_hdr_info, _, _ = parse_envi_header(WHITE_HDR)
        report["white_reference_header"] = white_hdr_info
        print(f"  White ref shape         : {white_hdr_info['lines']} × {white_hdr_info['samples']} × {white_hdr_info['bands']}")
    if DARK_HDR.exists():
        dark_hdr_info, _, _ = parse_envi_header(DARK_HDR)
        report["dark_reference_header"] = dark_hdr_info

    # -- Load cubes ---------------------------------------------------------
    print("\n--- Loading Hyperspectral Cubes ---")
    print("  Loading raw cube ... (this may take a moment for a 1.2 GB file)")
    raw_cube = load_cube(RAW_HDR)
    print(f"  Raw cube loaded  : shape={raw_cube.shape}  dtype={raw_cube.dtype}")
    report["raw_cube_shape"] = list(raw_cube.shape)
    report["raw_cube_dtype"] = str(raw_cube.dtype)

    white_cube = load_cube(WHITE_HDR) if WHITE_HDR.exists() else None
    dark_cube  = load_cube(DARK_HDR)  if DARK_HDR.exists()  else None

    # -- Calibration --------------------------------------------------------
    cal_cube = raw_cube
    if white_cube is not None and dark_cube is not None:
        print("  Applying white/dark calibration ...")
        # References are (lines=1, samples, bands) — broadcast over lines
        white_ref = white_cube.mean(axis=0, keepdims=True)
        dark_ref  = dark_cube.mean(axis=0, keepdims=True)
        cal_cube  = calibrate(raw_cube, white_ref, dark_ref)
        print(f"  Calibrated cube  : min={cal_cube.min():.4f}  max={cal_cube.max():.4f}")
        report["calibration"] = "white/dark reflectance normalisation applied"
    else:
        print("  [WARN] Missing white/dark references — using raw values")
        report["calibration"] = "none (missing references)"

    # -- Pixel statistics ---------------------------------------------------
    print("\n--- Pixel Statistics ---")
    h, w, b = cal_cube.shape
    flat = cal_cube.reshape(-1, b)
    print(f"  Full cube stats  : mean={flat.mean():.4f}  std={flat.std():.4f}  "
          f"min={flat.min():.4f}  max={flat.max():.4f}")
    report["cube_stats"] = {
        "shape": list(cal_cube.shape),
        "mean":  round(float(flat.mean()), 6),
        "std":   round(float(flat.std()), 6),
        "min":   round(float(flat.min()), 6),
        "max":   round(float(flat.max()), 6),
    }

    # -- Band statistics (sampled) ------------------------------------------
    print("\n--- Per-Band Statistics (20 sampled bands) ---")
    bstats = band_statistics(cal_cube, wavelengths, n_sample=20)
    report["band_statistics_sampled"] = bstats
    print(f"  {'Band':>5}  {'WL(nm)':>8}  {'Mean':>8}  {'Std':>8}")
    for bs in bstats:
        print(f"  {bs['band_index']:>5}  {bs['wavelength_nm']:>8.1f}  "
              f"{bs['mean']:>8.5f}  {bs['std']:>8.5f}")

    # -- Spectral signatures ------------------------------------------------
    print("\n--- Spectral Signatures at Representative Pixels ---")
    pixels = [("center", h // 2, w // 2), ("top_left", h // 8, w // 8),
              ("mid_right", h // 2, 7 * w // 8)]
    sigs = {}
    for name, r, c in pixels:
        sig = spectral_signature(cal_cube, r, c, wavelengths)
        sigs[name] = sig
        print(f"  Pixel [{name:10s}] ({r},{c})  peak={sig['peak_wl_nm']}nm  "
              f"mean={sig['mean']:.5f}")
    report["spectral_signatures"] = sigs

    # -- SDK demo -----------------------------------------------------------
    sdk_info = sdk_rgb_demo(RGB_PREVIEW)
    if sdk_info:
        report["sdk_simple_wrapper_rgb"] = sdk_info

    # -- Visualization ------------------------------------------------------
    make_visualization(raw_cube, cal_cube, wavelengths, RGB_PREVIEW, sdk_info)

    # -- Save report --------------------------------------------------------
    report_path = OUTPUT_DIR / f"{PATIENT_ID}_{ROI_ID}_report.json"
    with open(report_path, "w") as f:
        json.dump(report, f, indent=2)

    print(f"\n[REPORT] Saved → {report_path}")
    print("\n[DONE]")


if __name__ == "__main__":
    main()


## Test: `test_quilt1m_laion.py`
**Documentation**:
```text
Test program: Quilt1M — LAION (web-scraped) subset
Sample     : 00004000040081.jpg
Source     : LAION large-scale web-scraped image-text dataset
Caption    : Cytosolic Sulfotransferase 1A1 / SULT1A1 antibody

Loads the sample image, extracts pixel statistics, retrieves full CSV metadata
(caption, UMLS entities, pathology sub-specialty, magnification), renders a
diagnostic visualization, and writes a JSON report.
```

**Expected Input:** Sample image file loaded from the relative `week1/data` directory as resolved below.
**Expected Output:** Successful execution outputting pixel metrics and dumping JSON reports/matplotlib figures into `week1/output`.


In [ ]:
"""
Test program: Quilt1M — LAION (web-scraped) subset
Sample     : 00004000040081.jpg
Source     : LAION large-scale web-scraped image-text dataset
Caption    : Cytosolic Sulfotransferase 1A1 / SULT1A1 antibody

Loads the sample image, extracts pixel statistics, retrieves full CSV metadata
(caption, UMLS entities, pathology sub-specialty, magnification), renders a
diagnostic visualization, and writes a JSON report.
"""

from pathlib import Path
from _quilt1m_common import run_test

SUBSET           = "laion"
IMAGE_FILENAME   = "00004000040081.jpg"
DATA_DIR         = Path("week1/test_quilt1m_laion.py").parent / "data" / "Quilt1M_laion"
OUTPUT_DIR       = Path("week1/test_quilt1m_laion.py").parent / "output" / "Quilt1M_laion"

if __name__ == "__main__":
    run_test(SUBSET, IMAGE_FILENAME, DATA_DIR, OUTPUT_DIR)


## Test: `test_quilt1m_openpath.py`
**Documentation**:
```text
Test program: Quilt1M — OpenPath (Twitter/social media) subset
Sample     : 994701482116173824_0.jpg
Source     : Twitter/social media pathology post (OpenPath collection)
Caption    : Peritoneal biopsy — DailyDx challenge (UMichPath / SurgPath)

Loads the sample image, extracts pixel statistics, retrieves full CSV metadata
(caption, UMLS entities, pathology sub-specialty, magnification), renders a
diagnostic visualization, and writes a JSON report.
```

**Expected Input:** Sample image file loaded from the relative `week1/data` directory as resolved below.
**Expected Output:** Successful execution outputting pixel metrics and dumping JSON reports/matplotlib figures into `week1/output`.


In [ ]:
"""
Test program: Quilt1M — OpenPath (Twitter/social media) subset
Sample     : 994701482116173824_0.jpg
Source     : Twitter/social media pathology post (OpenPath collection)
Caption    : Peritoneal biopsy — DailyDx challenge (UMichPath / SurgPath)

Loads the sample image, extracts pixel statistics, retrieves full CSV metadata
(caption, UMLS entities, pathology sub-specialty, magnification), renders a
diagnostic visualization, and writes a JSON report.
"""

from pathlib import Path
from _quilt1m_common import run_test

SUBSET           = "openpath"
IMAGE_FILENAME   = "994701482116173824_0.jpg"
DATA_DIR         = Path("week1/test_quilt1m_openpath.py").parent / "data" / "Quilt1M_openpath"
OUTPUT_DIR       = Path("week1/test_quilt1m_openpath.py").parent / "output" / "Quilt1M_openpath"

if __name__ == "__main__":
    run_test(SUBSET, IMAGE_FILENAME, DATA_DIR, OUTPUT_DIR)


## Test: `test_quilt1m_pubmed.py`
**Documentation**:
```text
Test program: Quilt1M — PubMed subset
Sample     : c901a42b-0ab9-45d9-809d-dd646effcf9c_1.jpg
Source     : PubMed open-access figure (academic paper illustration)
Caption    : IDH1 immunocytochemistry in MG63 and U2OS osteosarcoma cell lines

Loads the sample image, extracts pixel statistics, retrieves full CSV metadata
(caption, UMLS entities, pathology sub-specialty, magnification), renders a
diagnostic visualization, and writes a JSON report.
```

**Expected Input:** Sample image file loaded from the relative `week1/data` directory as resolved below.
**Expected Output:** Successful execution outputting pixel metrics and dumping JSON reports/matplotlib figures into `week1/output`.


In [ ]:
"""
Test program: Quilt1M — PubMed subset
Sample     : c901a42b-0ab9-45d9-809d-dd646effcf9c_1.jpg
Source     : PubMed open-access figure (academic paper illustration)
Caption    : IDH1 immunocytochemistry in MG63 and U2OS osteosarcoma cell lines

Loads the sample image, extracts pixel statistics, retrieves full CSV metadata
(caption, UMLS entities, pathology sub-specialty, magnification), renders a
diagnostic visualization, and writes a JSON report.
"""

from pathlib import Path
from _quilt1m_common import run_test

SUBSET           = "pubmed"
IMAGE_FILENAME   = "c901a42b-0ab9-45d9-809d-dd646effcf9c_1.jpg"
DATA_DIR         = Path("week1/test_quilt1m_pubmed.py").parent / "data" / "Quilt1M_pubmed"
OUTPUT_DIR       = Path("week1/test_quilt1m_pubmed.py").parent / "output" / "Quilt1M_pubmed"

if __name__ == "__main__":
    run_test(SUBSET, IMAGE_FILENAME, DATA_DIR, OUTPUT_DIR)


## Test: `test_quilt1m_quilt.py`
**Documentation**:
```text
Test program: Quilt1M — Quilt (YouTube) subset
Sample     : dTr3MNl1FxE_image_c54e9a8d-9348-456a-9645-3b8921eb0b79.jpg
Source     : Frame extraction from YouTube pathology educational video
Caption    : Nephrogenic systemic fibrosis — dermatopathology / soft tissue / breast

Loads the sample image, extracts pixel statistics, retrieves full CSV metadata
(caption, UMLS entities, pathology sub-specialty, magnification), renders a
diagnostic visualization, and writes a JSON report.
```

**Expected Input:** Sample image file loaded from the relative `week1/data` directory as resolved below.
**Expected Output:** Successful execution outputting pixel metrics and dumping JSON reports/matplotlib figures into `week1/output`.


In [ ]:
"""
Test program: Quilt1M — Quilt (YouTube) subset
Sample     : dTr3MNl1FxE_image_c54e9a8d-9348-456a-9645-3b8921eb0b79.jpg
Source     : Frame extraction from YouTube pathology educational video
Caption    : Nephrogenic systemic fibrosis — dermatopathology / soft tissue / breast

Loads the sample image, extracts pixel statistics, retrieves full CSV metadata
(caption, UMLS entities, pathology sub-specialty, magnification), renders a
diagnostic visualization, and writes a JSON report.
"""

from pathlib import Path
from _quilt1m_common import run_test

SUBSET           = "quilt"
IMAGE_FILENAME   = "dTr3MNl1FxE_image_c54e9a8d-9348-456a-9645-3b8921eb0b79.jpg"
DATA_DIR         = Path("week1/test_quilt1m_quilt.py").parent / "data" / "Quilt1M_quilt"
OUTPUT_DIR       = Path("week1/test_quilt1m_quilt.py").parent / "output" / "Quilt1M_quilt"

if __name__ == "__main__":
    run_test(SUBSET, IMAGE_FILENAME, DATA_DIR, OUTPUT_DIR)


## Test: `test_spinal_dicom.py`
**Documentation**:
```text
Test program: Spinal Multiple Myeloma — DICOM (Spectral CT)
Modality   : Spectral/Dual-Energy CT (Spine)
Domain     : Radiology (Oncology — Multiple Myeloma)
Format     : DICOM (.dcm) — MonoE 80 keV reconstruction, Myel_001
Tools      : DicomImageWrapper (TissueLab-SDK), pydicom (full tag dump),
             SimpleITK (3D series reconstruction), pandas (metadata.csv),
             matplotlib, numpy

What this program extracts:
  - Full DICOM tag dump of the first slice (pydicom)
  - Key clinical DICOM tags (patient, scanner, acquisition, reconstruction)
  - DicomImageWrapper region read (SDK interface demo)
  - SimpleITK 3D volume reconstruction from the series (shape, voxel spacing, HU range)
  - Three-plane volume visualisation (axial / coronal / sagittal)
  - Intensity statistics and HU histogram
  - metadata.csv row for Myel_001
  - JSON + PNG report output

Note: Only 20 DICOM slices are copied to week1/data for portability.
      The full 1095-slice series lives in the original Spinal dataset.
```

**Expected Input:** Sample image file loaded from the relative `week1/data` directory as resolved below.
**Expected Output:** Successful execution outputting pixel metrics and dumping JSON reports/matplotlib figures into `week1/output`.


In [ ]:
"""
Test program: Spinal Multiple Myeloma — DICOM (Spectral CT)
Modality   : Spectral/Dual-Energy CT (Spine)
Domain     : Radiology (Oncology — Multiple Myeloma)
Format     : DICOM (.dcm) — MonoE 80 keV reconstruction, Myel_001
Tools      : DicomImageWrapper (TissueLab-SDK), pydicom (full tag dump),
             SimpleITK (3D series reconstruction), pandas (metadata.csv),
             matplotlib, numpy

What this program extracts:
  - Full DICOM tag dump of the first slice (pydicom)
  - Key clinical DICOM tags (patient, scanner, acquisition, reconstruction)
  - DicomImageWrapper region read (SDK interface demo)
  - SimpleITK 3D volume reconstruction from the series (shape, voxel spacing, HU range)
  - Three-plane volume visualisation (axial / coronal / sagittal)
  - Intensity statistics and HU histogram
  - metadata.csv row for Myel_001
  - JSON + PNG report output

Note: Only 20 DICOM slices are copied to week1/data for portability.
      The full 1095-slice series lives in the original Spinal dataset.
"""

import json
import sys
from pathlib import Path

import numpy as np

try:
    import pydicom
    HAS_PYDICOM = True
except ImportError:
    print("[ERROR] pydicom is required: pip install pydicom")
    sys.exit(1)

try:
    import SimpleITK as sitk
    HAS_SITK = True
except ImportError:
    HAS_SITK = False
    print("[WARN] SimpleITK not installed — 3D reconstruction unavailable")

try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False
    print("[WARN] pandas not installed — metadata.csv lookup unavailable")

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("[WARN] matplotlib not available — skipping visualizations")

try:
    from tissuelab_sdk.wrapper import DicomImageWrapper
    HAS_SDK = True
except ImportError:
    HAS_SDK = False
    print("[WARN] TissueLab-SDK not installed — skipping DicomImageWrapper demo")

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
PATIENT_ID   = "Myel_001"
SERIES_NAME  = "MonoE_80keVHU"

DATA_DIR     = Path("week1/test_spinal_dicom.py").parent / "data" / "Spinal_DICOM"
SERIES_DIR   = DATA_DIR / PATIENT_ID / SERIES_NAME
METADATA_CSV = DATA_DIR / "metadata.csv"
OUTPUT_DIR   = Path("week1/test_spinal_dicom.py").parent / "output" / "Spinal_DICOM"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Full-series path (for SimpleITK — if the full copy exists)
FULL_SERIES_DIR = Path(
    "/Volumes/ExternalOwc/AI_For_Healthcare/Final_Project/Datasets"
    "/Spinal/manifest-1774389300184/Spinal-Multiple-Myeloma-SEG"
    f"/Myel_001/01-06-2015-8006-NA-94443/20781.000000-MonoE 80keVHU-77135"
)

# ---------------------------------------------------------------------------
# DICOM tag extraction
# ---------------------------------------------------------------------------
CLINICAL_TAGS = {
    "PatientID":                "0010,0020",
    "StudyDate":                "0008,0020",
    "Modality":                 "0008,0060",
    "SeriesDescription":        "0008,103e",
    "Manufacturer":             "0008,0070",
    "ManufacturerModelName":    "0008,1090",
    "KVP":                      "0018,0060",
    "SliceThickness":           "0018,0050",
    "PixelSpacing":             "0028,0030",
    "Rows":                     "0028,0010",
    "Columns":                  "0028,0011",
    "BitsAllocated":            "0028,0100",
    "RescaleIntercept":         "0028,1052",
    "RescaleSlope":             "0028,1053",
    "ImagePositionPatient":     "0020,0032",
    "ImageOrientationPatient":  "0020,0037",
    "SliceLocation":            "0020,1041",
    "InstanceNumber":           "0020,0013",
    "PhotometricInterpretation":"0028,0004",
    "WindowCenter":             "0028,1050",
    "WindowWidth":              "0028,1051",
}


def extract_tag_value(ds: pydicom.Dataset, tag_name: str) -> str:
    """Safely extract a DICOM tag value by keyword name."""
    try:
        val = getattr(ds, tag_name, None)
        if val is None:
            return "N/A"
        if hasattr(val, "__iter__") and not isinstance(val, str):
            return str(list(val))
        return str(val)
    except Exception:
        return "N/A"


def dump_all_tags(ds: pydicom.Dataset) -> dict:
    """Return a dict of all non-pixel-data DICOM elements."""
    tags = {}
    for elem in ds:
        if elem.keyword == "PixelData":
            continue
        try:
            tags[elem.keyword or str(elem.tag)] = str(elem.value)[:200]
        except Exception:
            tags[str(elem.tag)] = "<unreadable>"
    return tags


def to_hounsfield(pixel_array: np.ndarray, ds: pydicom.Dataset) -> np.ndarray:
    """Apply RescaleSlope and RescaleIntercept to get HU values."""
    slope     = float(getattr(ds, "RescaleSlope",     1.0) or 1.0)
    intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
    return pixel_array.astype(float) * slope + intercept


# ---------------------------------------------------------------------------
# Metadata CSV
# ---------------------------------------------------------------------------
def lookup_metadata_csv(csv_path: Path, subject_id: str) -> list[dict]:
    if not HAS_PANDAS or not csv_path.exists():
        return []
    df = pd.read_csv(csv_path)
    matches = df[df["Subject ID"] == subject_id]
    return matches.to_dict(orient="records")


# ---------------------------------------------------------------------------
# SimpleITK 3D reconstruction
# ---------------------------------------------------------------------------
def sitk_load_series(series_dir: Path) -> dict | None:
    if not HAS_SITK:
        return None
    if not series_dir.exists():
        print(f"  [WARN] Series directory not found for SimpleITK: {series_dir}")
        return None

    print(f"\n  [SimpleITK] Reading series from: {series_dir}")
    reader = sitk.ImageSeriesReader()
    dcm_names = reader.GetGDCMSeriesFileNames(str(series_dir))
    if not dcm_names:
        print("  [WARN] No DICOM files found in series directory")
        return None

    print(f"  Found {len(dcm_names)} DICOM files")
    reader.SetFileNames(dcm_names)
    image = reader.Execute()

    size    = image.GetSize()          # (x, y, z)
    spacing = image.GetSpacing()       # mm per voxel (x, y, z)
    origin  = image.GetOrigin()
    direction = image.GetDirection()

    arr = sitk.GetArrayFromImage(image)  # shape: (z, y, x)
    print(f"  Volume shape (z,y,x) : {arr.shape}")
    print(f"  Voxel spacing (mm)   : {[round(s, 4) for s in spacing]}")
    print(f"  HU range             : {arr.min():.1f} – {arr.max():.1f}")

    return {
        "sitk_size_xyz":    list(size),
        "volume_shape_zyx": list(arr.shape),
        "voxel_spacing_mm": [round(float(s), 4) for s in spacing],
        "origin_mm":        [round(float(o), 4) for o in origin],
        "n_files_loaded":   len(dcm_names),
        "hu_min":           float(arr.min()),
        "hu_max":           float(arr.max()),
        "hu_mean":          round(float(arr.mean()), 2),
        "hu_std":           round(float(arr.std()), 2),
        "array":            arr,   # not serialised; used for viz
    }


# ---------------------------------------------------------------------------
# Visualization
# ---------------------------------------------------------------------------
def make_visualization(
    first_slice_hu: np.ndarray,
    volume_info: dict | None,
    all_hu: list[np.ndarray],
) -> None:
    if not HAS_MPL:
        return

    n_panels = 3 if volume_info else 2
    fig, axes = plt.subplots(1, n_panels + 1, figsize=(5 * (n_panels + 1), 5))

    # Panel 0: First DICOM slice (windowed for soft tissue)
    wc, ww = 40, 400  # typical soft tissue window
    vmin, vmax = wc - ww / 2, wc + ww / 2
    axes[0].imshow(first_slice_hu, cmap="gray", vmin=vmin, vmax=vmax, aspect="equal")
    axes[0].set_title(f"Slice 1 — MonoE 80 keV\n(windowed W={ww}/L={wc})", fontsize=10)
    axes[0].axis("off")

    if volume_info and "array" in volume_info:
        vol = volume_info["array"]
        nz, ny, nx = vol.shape
        # Axial (mid)
        axes[1].imshow(vol[nz // 2, :, :], cmap="gray", vmin=vmin, vmax=vmax, aspect="equal")
        axes[1].set_title(f"Axial mid-slice\n({nz} total slices)", fontsize=10)
        axes[1].axis("off")
        # Coronal (mid)
        axes[2].imshow(vol[:, ny // 2, :], cmap="gray", vmin=vmin, vmax=vmax, aspect="equal")
        axes[2].set_title("Coronal mid-slice", fontsize=10)
        axes[2].axis("off")

    # Last panel: HU histogram
    ax_hist = axes[-1]
    if all_hu:
        hu_flat = np.concatenate([s.flatten() for s in all_hu])
        ax_hist.hist(hu_flat, bins=80, color="steelblue", edgecolor="none", alpha=0.8)
        ax_hist.set_xlabel("HU value", fontsize=9)
        ax_hist.set_ylabel("Voxel count", fontsize=9)
        ax_hist.set_title("HU distribution\n(all loaded slices)", fontsize=10)
        ax_hist.axvline(-100, color="red",   linestyle="--", alpha=0.7, label="~Fat")
        ax_hist.axvline(  40, color="green", linestyle="--", alpha=0.7, label="~Soft tissue")
        ax_hist.axvline( 400, color="orange",linestyle="--", alpha=0.7, label="~Bone")
        ax_hist.legend(fontsize=7)
        ax_hist.grid(True, alpha=0.3)

    fig.suptitle(
        f"Spinal Multiple Myeloma — {PATIENT_ID}\n"
        "Spectral CT (MonoE 80 keV) — Spine + Lesion imaging",
        fontsize=13, fontweight="bold"
    )
    out_path = OUTPUT_DIR / f"{PATIENT_ID}_DICOM_analysis.png"
    fig.savefig(str(out_path), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"\n  [VIZ] Saved → {out_path}")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def main() -> None:
    print("\n" + "="*60)
    print("  Spinal Multiple Myeloma — DICOM Probe")
    print(f"  Patient  : {PATIENT_ID}")
    print(f"  Series   : {SERIES_NAME}")
    print("  Modality : Spectral CT (Dual-Energy) — Spine")
    print("  Domain   : Radiology / Oncology")
    print("="*60)

    report: dict = {
        "dataset":   "Spinal-Multiple-Myeloma-SEG",
        "patient":   PATIENT_ID,
        "series":    SERIES_NAME,
        "modality":  "Spectral CT (Dual-Energy) — Spine",
        "domain":    "Radiology / Oncology",
        "clinical_context": (
            "67-patient DECT spine dataset (Philips IQon). Multiple Myeloma lesion "
            "segmentation task. MonoE 80 keV is a standard soft-tissue equivalent "
            "reconstruction. Lesion and spine NIfTI masks available separately."
        ),
    }

    # -- Find DICOM slices --------------------------------------------------
    dcm_files = sorted(SERIES_DIR.glob("*.dcm"))
    if not dcm_files:
        print(f"  [ERROR] No .dcm files found in {SERIES_DIR}")
        sys.exit(1)

    print(f"\n  Found {len(dcm_files)} DICOM slice(s) in local copy ({SERIES_DIR.name})")
    report["local_slice_count"] = len(dcm_files)

    # -- Load first slice with pydicom --------------------------------------
    first_dcm = dcm_files[0]
    print(f"\n--- pydicom: First Slice ({first_dcm.name}) ---")
    ds = pydicom.dcmread(str(first_dcm))

    all_tags = dump_all_tags(ds)
    report["all_dicom_tags_slice1"] = all_tags

    print("\n  Key Clinical DICOM Tags:")
    clinical = {}
    for name in CLINICAL_TAGS:
        val = extract_tag_value(ds, name)
        clinical[name] = val
        print(f"    {name:<30}: {val}")
    report["clinical_tags"] = clinical

    # HU conversion for the first slice
    pixel_arr = ds.pixel_array
    hu_slice  = to_hounsfield(pixel_arr, ds)
    print(f"\n  Pixel array shape : {pixel_arr.shape}")
    print(f"  HU range          : {hu_slice.min():.1f} – {hu_slice.max():.1f}")
    print(f"  HU mean / std     : {hu_slice.mean():.2f} / {hu_slice.std():.2f}")
    report["first_slice_stats"] = {
        "pixel_shape": list(pixel_arr.shape),
        "hu_min":      round(float(hu_slice.min()), 2),
        "hu_max":      round(float(hu_slice.max()), 2),
        "hu_mean":     round(float(hu_slice.mean()), 2),
        "hu_std":      round(float(hu_slice.std()), 2),
    }

    # -- SDK DicomImageWrapper demo -----------------------------------------
    all_hu_slices = [hu_slice]
    if HAS_SDK:
        print(f"\n  [SDK] Loading via DicomImageWrapper: {first_dcm.name}")
        try:
            wrapper = DicomImageWrapper(str(first_dcm))
            w, h    = wrapper.dimensions
            region  = wrapper.read_region((0, 0), 0, (w, h), as_array=True)
            sdk_info = {
                "wrapper_class":     type(wrapper).__name__,
                "dimensions_wh":     list(wrapper.dimensions),
                "properties":        dict(wrapper.properties),
                "region_shape":      list(region.shape),
            }
            report["sdk_dicom_wrapper"] = sdk_info
            print(f"    Dimensions : {sdk_info['dimensions_wh']}")
            print(f"    Properties : {sdk_info['properties']}")
            wrapper.close()
        except Exception as e:
            print(f"    [WARN] DicomImageWrapper error: {e}")
            report["sdk_dicom_wrapper"] = {"error": str(e)}

    # -- Load all local slices for histogram --------------------------------
    print(f"\n--- Loading all {len(dcm_files)} local DICOM slices ---")
    for dcm_path in dcm_files[1:]:
        ds_i = pydicom.dcmread(str(dcm_path))
        hu_i = to_hounsfield(ds_i.pixel_array, ds_i)
        all_hu_slices.append(hu_i)

    all_hu_arr = np.stack(all_hu_slices, axis=0)
    print(f"  Stacked local volume : {all_hu_arr.shape}  "
          f"HU={all_hu_arr.min():.0f}–{all_hu_arr.max():.0f}")
    report["local_volume_stats"] = {
        "shape":    list(all_hu_arr.shape),
        "hu_min":   float(all_hu_arr.min()),
        "hu_max":   float(all_hu_arr.max()),
        "hu_mean":  round(float(all_hu_arr.mean()), 2),
        "hu_std":   round(float(all_hu_arr.std()), 2),
    }

    # -- SimpleITK full series (uses original path) -------------------------
    volume_info = None
    if HAS_SITK:
        # Prefer full series if available; fall back to local copy
        series_for_sitk = FULL_SERIES_DIR if FULL_SERIES_DIR.exists() else SERIES_DIR
        volume_info = sitk_load_series(series_for_sitk)
        if volume_info:
            arr = volume_info.pop("array", None)    # remove non-serialisable
            report["sitk_3d_volume"] = volume_info
            volume_info["array"] = arr              # restore for viz

    # -- Metadata CSV -------------------------------------------------------
    print("\n--- metadata.csv ---")
    csv_rows = lookup_metadata_csv(METADATA_CSV, PATIENT_ID)
    if csv_rows:
        print(f"  Found {len(csv_rows)} series for {PATIENT_ID}")
        for row in csv_rows[:3]:
            print(f"    {row.get('Series Description','?'):<30}  "
                  f"images={row.get('Number of Images','?')}")
        report["metadata_csv_rows"] = csv_rows
    else:
        print("  No metadata rows found")

    # -- Visualization ------------------------------------------------------
    make_visualization(hu_slice, volume_info, all_hu_slices)

    # -- Save report --------------------------------------------------------
    report_path = OUTPUT_DIR / f"{PATIENT_ID}_DICOM_report.json"
    with open(report_path, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\n[REPORT] Saved → {report_path}")
    print("\n[DONE]")


if __name__ == "__main__":
    main()


## Test: `test_spinal_nifti.py`
**Documentation**:
```text
Test program: Spinal Multiple Myeloma — NIfTI Segmentation Masks
Modality   : Spectral CT derived segmentation (spine anatomy + lesion masks)
Domain     : Radiology / Oncology (Multiple Myeloma)
Format     : NIfTI (.nii) — uncompressed
Tools      : NiftiImageWrapper (TissueLab-SDK), nibabel (full header + volume),
             numpy, matplotlib, pandas (metadata.csv)

Sample     : Myel_001
  - Myel_001_spine_segmentation.nii   — vertebra-level spine anatomy mask
  - Myel_001_lesions_segmentation.nii — lesion (tumour) mask

What this program extracts:
  - NIfTI header metadata (voxel dimensions, orientation, data type, affine)
  - Per-label voxel counts and volume (cm³) for each mask
  - Unique label values (vertebra IDs for spine mask, lesion IDs for lesion mask)
  - NiftiImageWrapper region read (SDK interface demo — sagittal mid-slice)
  - Three-plane visualizations: spine mask + lesion mask overlay
  - Spatial co-registration check (same FOV between masks)
  - metadata.csv row for Myel_001
  - JSON + PNG report output
```

**Expected Input:** Sample image file loaded from the relative `week1/data` directory as resolved below.
**Expected Output:** Successful execution outputting pixel metrics and dumping JSON reports/matplotlib figures into `week1/output`.


In [ ]:
"""
Test program: Spinal Multiple Myeloma — NIfTI Segmentation Masks
Modality   : Spectral CT derived segmentation (spine anatomy + lesion masks)
Domain     : Radiology / Oncology (Multiple Myeloma)
Format     : NIfTI (.nii) — uncompressed
Tools      : NiftiImageWrapper (TissueLab-SDK), nibabel (full header + volume),
             numpy, matplotlib, pandas (metadata.csv)

Sample     : Myel_001
  - Myel_001_spine_segmentation.nii   — vertebra-level spine anatomy mask
  - Myel_001_lesions_segmentation.nii — lesion (tumour) mask

What this program extracts:
  - NIfTI header metadata (voxel dimensions, orientation, data type, affine)
  - Per-label voxel counts and volume (cm³) for each mask
  - Unique label values (vertebra IDs for spine mask, lesion IDs for lesion mask)
  - NiftiImageWrapper region read (SDK interface demo — sagittal mid-slice)
  - Three-plane visualizations: spine mask + lesion mask overlay
  - Spatial co-registration check (same FOV between masks)
  - metadata.csv row for Myel_001
  - JSON + PNG report output
"""

import json
import sys
from pathlib import Path

import numpy as np

try:
    import nibabel as nib
    HAS_NIBABEL = True
except ImportError:
    print("[ERROR] nibabel is required: pip install nibabel")
    sys.exit(1)

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import matplotlib.colors as mcolors
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("[WARN] matplotlib not installed — skipping visualizations")

try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False

try:
    from tissuelab_sdk.wrapper import NiftiImageWrapper
    HAS_SDK = True
except ImportError:
    HAS_SDK = False
    print("[WARN] TissueLab-SDK not installed — skipping NiftiImageWrapper demo")

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
PATIENT_ID   = "Myel_001"
DATA_DIR     = Path("week1/test_spinal_nifti.py").parent / "data" / "Spinal_NIfTI" / PATIENT_ID
METADATA_CSV = Path("week1/test_spinal_nifti.py").parent / "data" / "Spinal_NIfTI" / "metadata.csv"
OUTPUT_DIR   = Path("week1/test_spinal_nifti.py").parent / "output" / "Spinal_NIfTI"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPINE_NII   = DATA_DIR / f"{PATIENT_ID}_spine_segmentation.nii"
LESION_NII  = DATA_DIR / f"{PATIENT_ID}_lesions_segmentation.nii"

# ---------------------------------------------------------------------------
# Header info
# ---------------------------------------------------------------------------
def nifti_header_info(img: nib.Nifti1Image, label: str) -> dict:
    hdr   = img.header
    zooms = hdr.get_zooms()
    shape = hdr.get_data_shape()
    info  = {
        "label":            label,
        "nifti_class":      type(img).__name__,
        "shape":            list(shape),
        "voxel_size_mm":    [round(float(z), 4) for z in zooms[:3]],
        "voxel_vol_mm3":    round(float(np.prod(zooms[:3])), 4),
        "data_dtype":       str(hdr.get_data_dtype()),
        "affine":           img.affine.tolist(),
        "qform_code":       int(hdr.get("qform_code", 0)),
        "sform_code":       int(hdr.get("sform_code", 0)),
        "description":      str(hdr.get("descrip", b"")).strip(),
    }
    print(f"\n  [{label}]")
    print(f"    Shape          : {info['shape']}")
    print(f"    Voxel size     : {info['voxel_size_mm']} mm")
    print(f"    Voxel volume   : {info['voxel_vol_mm3']:.4f} mm³")
    print(f"    Data dtype     : {info['data_dtype']}")
    return info


# ---------------------------------------------------------------------------
# Label analysis
# ---------------------------------------------------------------------------
def label_analysis(data: np.ndarray, mask_name: str, voxel_vol_mm3: float) -> dict:
    """Compute per-label statistics (voxel count, volume) for a segmentation mask."""
    unique_labels = sorted(np.unique(data).astype(int).tolist())
    label_stats: dict = {}
    print(f"\n  [{mask_name}] — unique labels: {unique_labels}")

    for lbl in unique_labels:
        voxels  = int((data == lbl).sum())
        vol_mm3 = round(voxels * voxel_vol_mm3, 2)
        vol_cm3 = round(vol_mm3 / 1000.0, 4)
        label_stats[str(lbl)] = {
            "voxels":   voxels,
            "vol_mm3":  vol_mm3,
            "vol_cm3":  vol_cm3,
        }
        if lbl != 0:  # skip background from printout
            tag = "Lesion" if mask_name == "Lesion" else f"Vertebra {lbl}"
            print(f"    Label {lbl:>3} ({tag:<16}): {voxels:>8,} voxels  "
                  f"({vol_mm3:>10.1f} mm³ / {vol_cm3:.4f} cm³)")

    bg = label_stats.get("0", {})
    fg_voxels = sum(v["voxels"] for k, v in label_stats.items() if k != "0")
    total     = int(data.size)
    print(f"    Background voxels : {bg.get('voxels', 0):>8,}")
    print(f"    Foreground voxels : {fg_voxels:>8,}  ({100*fg_voxels/total:.2f}% of volume)")

    return {
        "unique_labels":    unique_labels,
        "n_foreground_labels": len(unique_labels) - (1 if 0 in unique_labels else 0),
        "foreground_voxels": fg_voxels,
        "total_voxels":     total,
        "foreground_pct":   round(100 * fg_voxels / total, 4),
        "per_label":        label_stats,
    }


# ---------------------------------------------------------------------------
# SDK demo
# ---------------------------------------------------------------------------
def sdk_demo(nii_path: Path, label: str) -> dict | None:
    if not HAS_SDK:
        return None
    print(f"\n  [SDK] NiftiImageWrapper — {label}")
    try:
        wrapper = NiftiImageWrapper(str(nii_path))
        w, h    = wrapper.dimensions
        region  = wrapper.read_region((0, 0), 0, (w, h), as_array=True)
        info = {
            "wrapper_class":    type(wrapper).__name__,
            "dimensions_wh":    list(wrapper.dimensions),
            "level_count":      wrapper.level_count,
            "level_dimensions": [list(d) for d in wrapper.level_dimensions],
            "region_shape":     list(region.shape),
            "properties":       dict(wrapper.properties),
        }
        print(f"    Dimensions  : {info['dimensions_wh']}")
        print(f"    Levels      : {info['level_count']}")
        print(f"    Region arr  : {info['region_shape']}")
        wrapper.close()
        return info
    except Exception as e:
        print(f"    [WARN] Error: {e}")
        return {"error": str(e)}


# ---------------------------------------------------------------------------
# Visualization
# ---------------------------------------------------------------------------
def make_visualization(
    spine_data: np.ndarray,
    lesion_data: np.ndarray,
    spine_vox: list[float],
) -> None:
    if not HAS_MPL:
        return

    sx, sy, sz = spine_data.shape

    # Mid-slices
    sag_s = spine_data[sx // 2, :, :]
    cor_s = spine_data[:, sy // 2, :]
    axi_s = spine_data[:, :, sz // 2]

    sag_l = lesion_data[sx // 2, :, :]
    cor_l = lesion_data[:, sy // 2, :]
    axi_l = lesion_data[:, :, sz // 2]

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    # Spine mask colormap (one color per vertebra label)
    n_spine_labels = int(spine_data.max()) + 1
    spine_cmap = plt.cm.get_cmap("tab20", max(n_spine_labels, 2))

    for col, (sag, cor, axi, title_suffix) in enumerate([
        (sag_s, cor_s, axi_s, "Spine mask"),
        (sag_l, cor_l, axi_l, "Lesion mask"),
    ]):
        pass  # layout handled below
    # rows = Spine/Lesion, cols = Sag/Cor/Axi
    planes_spine  = [sag_s, cor_s, axi_s]
    planes_lesion = [sag_l, cor_l, axi_l]
    plane_names   = ["Sagittal (mid)", "Coronal (mid)", "Axial (mid)"]

    for col in range(3):
        # Row 0 — spine
        ax = axes[0, col]
        im = ax.imshow(np.rot90(planes_spine[col]),
                       cmap=spine_cmap, vmin=0, vmax=n_spine_labels,
                       interpolation="nearest")
        ax.set_title(f"Spine mask — {plane_names[col]}", fontsize=10)
        ax.axis("off")

        # Row 1 — lesion
        ax = axes[1, col]
        ax.imshow(np.rot90(planes_spine[col]),
                  cmap="gray_r", vmin=0, vmax=n_spine_labels,
                  interpolation="nearest", alpha=0.4)
        ax.imshow(np.rot90(planes_lesion[col]),
                  cmap="hot", vmin=0, vmax=max(1, int(lesion_data.max())),
                  interpolation="nearest", alpha=0.9)
        ax.set_title(f"Lesion mask overlay — {plane_names[col]}", fontsize=10)
        ax.axis("off")

    fig.suptitle(
        f"Spinal Multiple Myeloma — {PATIENT_ID}\n"
        "NIfTI Segmentation Masks: Spine Anatomy + Lesion",
        fontsize=13, fontweight="bold"
    )

    out_path = OUTPUT_DIR / f"{PATIENT_ID}_NIfTI_segmentation.png"
    fig.savefig(str(out_path), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"\n  [VIZ] Saved → {out_path}")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def main() -> None:
    print("\n" + "="*60)
    print("  Spinal Multiple Myeloma — NIfTI Segmentation Probe")
    print(f"  Patient  : {PATIENT_ID}")
    print("  Masks    : spine_segmentation + lesions_segmentation")
    print("  Modality : Spectral CT derived masks (NIfTI)")
    print("  Domain   : Radiology / Oncology")
    print("="*60)

    report: dict = {
        "dataset":   "Spinal-Multiple-Myeloma-SEG",
        "patient":   PATIENT_ID,
        "modality":  "NIfTI Segmentation Masks (Spectral CT derived)",
        "domain":    "Radiology / Oncology",
        "clinical_context": (
            "Expert-refined nnU-Net v2 segmentations. "
            "Spine mask labels = vertebra IDs (C1–L5 etc.). "
            "Lesion mask labels = Multiple Myeloma lesion foci. "
            "Source CT: Philips IQon DECT, spine + lesion annotations for 67 patients."
        ),
    }

    # -- Spine mask ---------------------------------------------------------
    print("\n--- Spine Segmentation Mask ---")
    if not SPINE_NII.exists():
        print(f"  [ERROR] Not found: {SPINE_NII}")
        sys.exit(1)

    spine_img  = nib.load(str(SPINE_NII))
    spine_hdr  = nifti_header_info(spine_img, "Spine mask")
    spine_data = np.asarray(spine_img.get_fdata(), dtype=int)
    spine_vox  = spine_hdr["voxel_size_mm"]
    spine_vv   = spine_hdr["voxel_vol_mm3"]

    spine_labels = label_analysis(spine_data, "Spine", spine_vv)
    report["spine_mask"] = {"header": spine_hdr, "label_analysis": spine_labels}

    # SDK demo — spine
    sdk_spine = sdk_demo(SPINE_NII, "Spine mask")
    if sdk_spine:
        report["sdk_nifti_wrapper_spine"] = sdk_spine

    # -- Lesion mask --------------------------------------------------------
    print("\n--- Lesion Segmentation Mask ---")
    if not LESION_NII.exists():
        print(f"  [ERROR] Not found: {LESION_NII}")
        sys.exit(1)

    lesion_img  = nib.load(str(LESION_NII))
    lesion_hdr  = nifti_header_info(lesion_img, "Lesion mask")
    lesion_data = np.asarray(lesion_img.get_fdata(), dtype=int)
    lesion_vv   = lesion_hdr["voxel_vol_mm3"]

    lesion_labels = label_analysis(lesion_data, "Lesion", lesion_vv)
    report["lesion_mask"] = {"header": lesion_hdr, "label_analysis": lesion_labels}

    # SDK demo — lesion
    sdk_lesion = sdk_demo(LESION_NII, "Lesion mask")
    if sdk_lesion:
        report["sdk_nifti_wrapper_lesion"] = sdk_lesion

    # -- Co-registration check ----------------------------------------------
    print("\n--- Co-registration Check ---")
    same_shape  = spine_data.shape == lesion_data.shape
    same_affine = np.allclose(spine_img.affine, lesion_img.affine, atol=1e-3)
    print(f"  Same shape   : {same_shape}  {spine_data.shape} vs {lesion_data.shape}")
    print(f"  Same affine  : {same_affine}")
    report["coregistration"] = {
        "same_shape":  same_shape,
        "same_affine": same_affine,
    }

    # -- Metadata CSV -------------------------------------------------------
    if HAS_PANDAS and METADATA_CSV.exists():
        df  = pd.read_csv(METADATA_CSV)
        rows = df[df["Subject ID"] == PATIENT_ID].to_dict(orient="records")
        print(f"\n--- metadata.csv: {len(rows)} series rows for {PATIENT_ID} ---")
        for row in rows[:3]:
            print(f"  {row.get('Series Description','?'):<35}  "
                  f"images={row.get('Number of Images','?')}")
        report["metadata_csv_rows"] = rows

    # -- Visualization ------------------------------------------------------
    make_visualization(spine_data, lesion_data, spine_vox)

    # -- Save report --------------------------------------------------------
    report_path = OUTPUT_DIR / f"{PATIENT_ID}_NIfTI_report.json"
    with open(report_path, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\n[REPORT] Saved → {report_path}")
    print("\n[DONE]")


if __name__ == "__main__":
    main()
